In [1]:
import tkinter as tk
from tkinter import filedialog
import cv2
import os
import numpy as np
import ctypes
from PIL import Image, ImageTk,ImageDraw
import module3

import smtplib
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import warnings
warnings.filterwarnings('ignore')
class OpenCVGUI:
    task=''
    file_path=''
    
    def __init__(self, master):
        self.master = master
        self.master.title("OpenCV GUI")
        self.master.geometry(f"{ctypes.windll.user32.GetSystemMetrics(0)}x{ctypes.windll.user32.GetSystemMetrics(1)}")

        # Create buttons
        self.image_button = tk.Button(master, text="Browse Image", command=self.browse_image)
        self.image_button.grid(column=0, row=0)

        self.video_button = tk.Button(master, text="Browse Video", command=self.browse_video)
        self.video_button.grid(column=1, row=0)

        self.camera_button = tk.Button(master, text="Live Camera", command=self.live_camera)
        self.camera_button.grid(column=2, row=0)
        self.process_button = tk.Button(master, text="Process", command=self.process)
        self.process_button.grid(column=3, row=0)

        # Create label for displaying video frames
        self.video_label = tk.Label(master)
        self.video_label.grid(column=2, row=2)
    
    
    
    def browse_image(self):
        self.task='image'
        # Open file dialog to select an image
        self.file_path = filedialog.askopenfilename(filetypes=[("Image files", "*.jpg;*.jpeg;*.png;*.bmp")])
        if self.file_path:
            # Read and display the image using OpenCV
            image = cv2.imread(self.file_path)
            self.display_image(image)

    def browse_video(self):
        self.task='video'
        # Open file dialog to select a video file
        self.file_path = filedialog.askopenfilename(filetypes=[("Video files", "*.mp4;*.avi")])
        if self.file_path:
            # Play the video using OpenCV
            cap = cv2.VideoCapture(self.file_path)
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                cv2.imshow("Video", frame)
                if cv2.waitKey(25) & 0xFF == ord('q'):
                    break
            cap.release()
            cv2.destroyAllWindows()


    def live_camera(self):
        self.task='live_camera'
        # Start live camera stream
        cap = cv2.VideoCapture(0)
        while True:
            ret, frame = cap.read()
            cv2.imwrite('input.jpg', frame)
            self.file_path='input.jpg'
            print('processing ',self.file_path)
            
            if not ret:
                break
            cv2.imshow("Live Camera", frame)
            clf = module3.main(self.file_path)
            print('activity:',clf) 
            if cv2.waitKey(25) & 0xFF == ord('q'):
                break
        cap.release()
        cv2.destroyAllWindows()

    def display_image(self, image):
        # Resize the image to 400x400
        image_resized = cv2.resize(image, (400, 400))
        # Convert the image from BGR to RGB
        image_rgb = cv2.cvtColor(image_resized, cv2.COLOR_BGR2RGB)
        # Convert the image to PIL format
        image_pil = Image.fromarray(image_rgb)
        # Convert PIL image to Tkinter PhotoImage
        photo = ImageTk.PhotoImage(image_pil)
        # Display the image in a label
        label = tk.Label(self.master, image=photo)
        label.image = photo
        label.grid(column=1, row=2)

    def play_video(self, file_path):
        # Open the video file
        cap = cv2.VideoCapture(file_path)
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                # Resize the frame to fit within 400x400
                frame_resized = cv2.resize(frame, (400, 400))
                self.display_frame(frame_resized)
                if cv2.waitKey(25) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        cv2.destroyAllWindows()

    def display_frame(self, frame):
        # Convert the frame from BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        # Convert the frame to PIL format
        frame_pil = Image.fromarray(frame_rgb)
        # Convert PIL image to Tkinter PhotoImage
        photo = ImageTk.PhotoImage(frame_pil)
        # Display the frame in the label
        self.video_label.config(image=photo)
        self.video_label.image = photo
        
        
    

    def send_email_with_attachment(self,sender_email, receiver_email, subject, body, attachment_path, gmail_password):
        # Create a multipart message
        msg = MIMEMultipart()
        msg['From'] = sender_email
        msg['To'] = receiver_email
        msg['Subject'] = subject

        # Add body to email
        msg.attach(MIMEText(body, 'plain'))

        # Open the file to be sent
        with open(attachment_path, 'rb') as attachment:
            # Add file as application/octet-stream
            part = MIMEBase('application', 'octet-stream')
            part.set_payload(attachment.read())

        # Encode file in ASCII characters to send by email    
        encoders.encode_base64(part)

        # Add header as key/value pair to attachment part
        part.add_header(
            'Content-Disposition',
            f'attachment; filename= {attachment_path}',
        )

        # Add attachment to message and convert message to string
        msg.attach(part)
        text = msg.as_string()

        # Log in to SMTP server
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(sender_email, gmail_password)

        # Send email
        server.sendmail(sender_email, receiver_email, text)
        server.quit()

    

    def video_to_frames(video, path_output_dir):
        vidcap = cv2.VideoCapture(video)
        count = 0
        while vidcap.isOpened():
            success, image = vidcap.read()
            if success:
                cv2.imwrite(os.path.join(path_output_dir, '%d.png') % count, image)
                count += 1
            else:
                break
        cv2.destroyAllWindows()
        vidcap.release()

    def process(self):
        print(self.task)
        if self.task=='':
            print('no input')
        elif self.task=='image':
            print('processing image',self.file_path)
            clf = module3.main(self.file_path)
            print('activity:',clf)  
            
        elif self.task=='video':
            print('processing video',self.file_path)
            video_to_frames(self.file_path,'input_images')
            files = []
            for file in os.listdir('input_images'):
                if os.path.isfile(os.path.join('input_images', file)):
                    files.append(file)
            for f in files:
                print('f',f)
                self.file_path='input_images/'+f
                print('processing ',self.file_path)
            
        
            
            
        

# Create Tkinter window
root = tk.Tk()
app = OpenCVGUI(root)
root.mainloop()


processing  input.jpg


usage: ipykernel_launcher.py [-h] [--model MODEL] [--label-bin LABEL_BIN] [--input INPUT] [--output OUTPUT] [-s SIZE]
ipykernel_launcher.py: error: unrecognized arguments: -f C:\Users\harsh\AppData\Roaming\jupyter\runtime\kernel-c7280df4-f7d9-4101-a502-56518fb9c21f.json


SystemExit: 2